In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

In [0]:
gold_wide = (
    silver
    .groupBy("country_code", "region", "year")
    .pivot("indicator_code")
    .agg(F.first("numeric_value"))
)

In [0]:
latest = spark.table("workspace.gold.gho_latest")
wide = spark.table("workspace.gold.gho_wide")

n_dup_latest = (latest.groupBy("country_code", "indicator_code").count()
    .filter("count > 1").count())
assert n_dup_latest == 0, f"{n_dup_latest} duplicated keys in gho_latest"

n_dup_wide = (wide.groupBy("country_code", "year").count()
    .filter("count > 1").count())
assert n_dup_wide == 0, f"{n_dup_wide} duplicated keys in gho_wide"

assert len(wide.columns) == 3 + 15, f"{len(wide.columns)} columns in gho_wide, expected 18"

In [0]:
indicator_names = {
    "GHED_GGHE-D_pc_US_SHA2011": "gov_health_spend_per_capita_usd",
    "HIV_0000000026": "hiv_indicator",
    "MDG_0000000029": "mdg_indicator",
    "NUTRITION_ANT_HAZ_NE2": "child_stunting_prev",
    "M_Est_smk_curr_std": "smoking_prev",
    "WHOSIS_000001": "life_expectancy",
    "WHOSIS_000002": "healthy_life_expectancy",
    "BP_04": "hypertension_prev",
    "NCDMORT3070": "ncd_mortality_30_70",
    "NCD_BMI_30C": "obesity_prev",
    "NCD_GLUC_04": "high_glucose_prev",
    "NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV": "anaemia_prev_women",
    "SDGPM25": "pm25_exposure",
    "WSH_WATER_SAFELY_MANAGED": "safe_water_pct",
    "WSH_SANITATION_SAFELY_MANAGED": "safe_sanitation_pct",
}

In [0]:
mapping = F.create_map([F.lit(x) for kv in indicator_names.items() for x in kv])
named = silver.withColumn("indicator", mapping[F.col("indicator_code")])

w = Window.partitionBy("country_code", "indicator").orderBy(F.desc("year"))
gold_latest = (
    named.withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn", "indicator_code")
)

gold_wide = (
    named.groupBy("country_code", "region", "year")
    .pivot("indicator", list(indicator_names.values()))
    .agg(F.first("numeric_value"))
)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

(gold_latest.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gho_latest"))

(gold_wide.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gho_wide"))